# visualization/01 — Spatial Figures

Produces the main spatial and temporal visualizations for ANTHEIA,
using *Achillea millefolium* as the case study species.

## Two visualization approaches

### Approach 1: Spatial interaction probability maps (predict_proba)
Runs the trained ANTHEIA-Scalar classifier across all CONUS bins and
computes per-bin mean P(interaction) for each week. Shows ground truth
(GloBI), Spatial Baseline (static), and ANTHEIA-Scalar (seasonal).

**Known limitation:** Logistic regression compresses all predictions
into a narrow high-probability range (mean ~0.88, std ~0.089). This
makes seasonal variation nearly invisible on a map — the model is
confident everywhere. The predict_proba maps are included for completeness
but are not the primary visualization.

### Approach 2: PPE Δ Hovmöller diagram (preferred)
Visualizes Δ = min(f_curve, a_curve) directly — the raw temporal overlap
signal before it enters the classifier. This avoids the logistic
compression problem and shows the seasonal and latitudinal structure
more clearly.

The Hovmöller diagram (latitude × week) shows the northward propagation
of peak temporal overlap through the season, concentrated at mid-latitudes
(35–45°N) between May and July. This is the figure used in the paper.

## Combined figure layout

**Row 0:** Ground Truth (GloBI) | Spatial Baseline (static) | PPE Δ across Spring / Summer / Fall / Winter

**Row 1:** Spatial Baseline latitude profile (static) | PPE Δ Hovmöller (latitude × week)

**Row 2:** Mean A3 − A2 difference map (full year)

In [ ]:
import numpy as np
import pandas as pd
import pickle
import glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from pathlib import Path

BASE   = Path("/scratch/ariana.l")
OLD_S4 = BASE / "Stage 4 Link Prediction Model"
NEW_S4 = BASE / "New Stage 4 Link Prediction Model"
PPE_DIR = BASE / "ppe-outputs" / "opportunity_surface"
OUT_DIR = OLD_S4 / "poster_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLANT   = "Achillea millefolium"
BIN_SIZE = 0.5
CONUS_LON = (-125, -66)
CONUS_LAT = (24, 50)

print("Paths OK")

In [ ]:
# Load existence matrices, embeddings, and trained models
print("Loading matrices and embeddings...")
F = pd.read_csv(OLD_S4 / "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(OLD_S4 / "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = [b for b in F.columns if b in set(P.columns)]
F_common = F[common_bins].values
P_common = P[common_bins].values
f_common_idx = {s: i for i, s in enumerate(F.index)}
p_common_idx = {s: i for i, s in enumerate(P.index)}

Vf_df = pd.read_csv(OLD_S4 / "stage4_Vf_phenofield.csv", index_col=0)
Vp_df = pd.read_csv(OLD_S4 / "stage4_Vp_gbif.csv", index_col=0)
Vf_arr = Vf_df.values
Vp_arr = Vp_df.values
vf_idx = {s: i for i, s in enumerate(Vf_df.index)}
vp_idx = {s: i for i, s in enumerate(Vp_df.index)}

print("Loading trained classifiers...")
with open(OLD_S4 / "stage4_A2_logistic.pkl", "rb") as f:
    clf_a2 = pickle.load(f)
with open(OLD_S4 / "stage4_A3_logistic.pkl", "rb") as f:
    clf_a3 = pickle.load(f)

print(f"Common bins: {len(common_bins)}")

In [ ]:
# Build flowering curves (f_curves) from PPE opportunity surface
# Average norm across all bins per (species, week)
print("Building flowering curves from PPE opportunity surface...")
files = sorted(glob.glob(str(PPE_DIR / "part_*.parquet")))

f_curves_dict = {}
for fpath in files:
    df = pd.read_parquet(fpath, columns=["species", "week", "norm"])
    species = df["species"].iloc[0]
    weekly_mean = df.groupby("week")["norm"].mean().reindex(range(52), fill_value=0)
    total = weekly_mean.sum()
    if total > 0:
        weekly_mean = weekly_mean / total
    f_curves_dict[species] = weekly_mean

flowering_curves = pd.DataFrame(f_curves_dict).T
print(f"  flowering_curves: {flowering_curves.shape}")

# Build activity curves (a_curves) from GBIF
print("Building activity curves from GBIF...")
GBIF_PATH = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
pol_doy = pd.read_csv(GBIF_PATH, usecols=["pollinator_species", "doy"], low_memory=False)
pol_doy = pol_doy.dropna(subset=["doy", "pollinator_species"])
pol_doy["week"] = ((pol_doy["doy"].astype(int) - 1) // 7).clip(0, 51)

week_counts = pol_doy.groupby(["pollinator_species", "week"]).size().unstack(fill_value=0)
activity_curves = week_counts.div(week_counts.sum(axis=1).replace(0, 1), axis=0)
print(f"  activity_curves: {activity_curves.shape}")

In [ ]:
# Load Achillea PPE opportunity surface and ground truth
print("Loading Achillea PPE surface...")
achillea_data = []
for fpath in files:
    df = pd.read_parquet(fpath)
    subset = df[df["species"] == PLANT]
    if len(subset) > 0:
        achillea_data.append(subset)
achillea = pd.concat(achillea_data, ignore_index=True)
print(f"  achillea: {achillea.shape}")

print("Loading ground truth...")
gt = pd.read_csv(OLD_S4 / "stage4_globi_conus_broad.csv")
gt = gt[
    (gt["sourceTaxonName"] == PLANT) |
    (gt["targetTaxonName"] == PLANT)
].dropna(subset=["decimalLatitude", "decimalLongitude"])
print(f"  ground truth obs: {len(gt):,}")

In [ ]:
# Build spatial grid and precompute A2 (Spatial Baseline — static)
print("Building spatial grid...")
vf_plant = Vf_arr[vf_idx[PLANT]]

# Bin → pollinator mapping
bin_to_pollinators = {}
for bin_col in P.columns:
    pols = P.index[P[bin_col] == 1].tolist()
    pols = [p for p in pols if p in vp_idx and p in activity_curves.index]
    if pols:
        bin_to_pollinators[bin_col] = pols

# Grid from Achillea PPE surface
achillea_grid = achillea[achillea["week"] == 0][["centroid_lat", "centroid_lon"]].drop_duplicates().copy()
achillea_grid["bin"] = (
    (np.floor(achillea_grid["centroid_lat"] / BIN_SIZE) * BIN_SIZE).round(1).astype(str) + "_" +
    (np.floor(achillea_grid["centroid_lon"] / BIN_SIZE) * BIN_SIZE).round(1).astype(str)
)

# Precompute A2 per bin (week-independent)
print("Precomputing Spatial Baseline per bin...")
bin_a2_cache = {}
bin_pol_N_cache = {}

for _, cell_row in achillea_grid.iterrows():
    bin_key = cell_row["bin"]
    if bin_key in bin_a2_cache:
        continue
    pols = bin_to_pollinators.get(bin_key, [])
    if not pols:
        bin_a2_cache[bin_key] = np.nan
        bin_pol_N_cache[bin_key] = []
        continue
    pol_N_list = []
    a2_preds = []
    for pol in pols:
        N_val = int(F_common[f_common_idx[PLANT]] @ P_common[p_common_idx[pol]]) \
                if PLANT in f_common_idx and pol in p_common_idx else 0
        feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val]])
        a2_preds.append(clf_a2.predict_proba(feat.reshape(1, -1))[0, 1])
        pol_N_list.append((pol, N_val))
    bin_a2_cache[bin_key] = np.mean(a2_preds)
    bin_pol_N_cache[bin_key] = pol_N_list

print(f"Grid cells: {len(achillea_grid):,}")

In [ ]:
# Compute A3 (ANTHEIA-Scalar) predict_proba and PPE Δ per bin per week
# Full year: weeks 0-51
print("Computing A3 predict_proba and PPE Δ per week (full year)...")
WEEKS = list(range(52))

pred_results  = []
delta_results = []

for week in WEEKS:
    if week % 8 == 0:
        print(f"  week {week}/51...")
    f_w = flowering_curves.loc[PLANT, week] if week in flowering_curves.columns else 0

    for _, cell_row in achillea_grid.iterrows():
        bin_key = cell_row["bin"]
        pol_N_list = bin_pol_N_cache.get(bin_key, [])
        lat, lon = cell_row["centroid_lat"], cell_row["centroid_lon"]

        if not pol_N_list:
            pred_results.append({"lat": lat, "lon": lon, "week": week,
                                  "pred_a2": np.nan, "pred_a3": np.nan})
            delta_results.append({"lat": lat, "lon": lon, "week": week, "delta": np.nan})
            continue

        a3_preds = []
        bin_deltas = []
        for pol, N_val in pol_N_list:
            a_w = activity_curves.loc[pol, week] if week in activity_curves.columns else 0
            delta = min(f_w, a_w)
            feat = np.concatenate([vf_plant, Vp_arr[vp_idx[pol]], [N_val, delta]])
            a3_preds.append(clf_a3.predict_proba(feat.reshape(1, -1))[0, 1])
            bin_deltas.append(delta)

        pred_results.append({"lat": lat, "lon": lon, "week": week,
                              "pred_a2": bin_a2_cache[bin_key],
                              "pred_a3": np.mean(a3_preds)})
        delta_results.append({"lat": lat, "lon": lon, "week": week,
                               "delta": np.mean(bin_deltas)})

pred_df  = pd.DataFrame(pred_results)
delta_df = pd.DataFrame(delta_results)

# Cache to parquet
pred_df.to_parquet(OLD_S4 / "pred_df_full_year.parquet",  index=False)
delta_df.to_parquet(OLD_S4 / "delta_df_full_year.parquet", index=False)
print(f"Done. pred_df: {pred_df.shape}, delta_df: {delta_df.shape}")
print()
print("Note: predict_proba is compressed (range ~0.6–1.0) because logistic")
print("regression is confident everywhere. Use PPE Δ for primary visualization.")

In [ ]:
# ── APPROACH 1: predict_proba spatial maps ────────────────────────────────────
# Included for completeness — shows the compression problem
# Four seasonal snapshots: Spring, Summer, Fall, Winter

SPATIAL_WEEKS = {12: "Spring (Wk12)", 25: "Summer (Wk25)", 38: "Fall (Wk38)", 51: "Winter (Wk51)"}
CMAP_PRED = "RdYlGn"

fig, axes = plt.subplots(1, len(SPATIAL_WEEKS) + 2, figsize=(22, 4), facecolor="white")

# Ground truth
axes[0].scatter(gt["decimalLongitude"], gt["decimalLatitude"],
                c="#2d6a4f", s=4, alpha=0.5)
axes[0].set_xlim(*CONUS_LON); axes[0].set_ylim(*CONUS_LAT)
axes[0].set_aspect("equal"); axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title("Ground Truth (GloBI)", fontsize=9, fontweight="bold")

# Spatial Baseline (static)
a2_data = pred_df[pred_df["week"] == 0]
axes[1].scatter(a2_data["lon"], a2_data["lat"],
                c=a2_data["pred_a2"], cmap=CMAP_PRED,
                s=4, vmin=0.6, vmax=1.0, alpha=0.8)
axes[1].set_xlim(*CONUS_LON); axes[1].set_ylim(*CONUS_LAT)
axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_title("Spatial Baseline\n(no temporal signal)", fontsize=9, fontweight="bold")

# ANTHEIA-Scalar per season
for i, (week, label) in enumerate(SPATIAL_WEEKS.items()):
    ax = axes[i + 2]
    week_data = pred_df[pred_df["week"] == week]
    sc = ax.scatter(week_data["lon"], week_data["lat"],
                    c=week_data["pred_a3"], cmap=CMAP_PRED,
                    s=4, vmin=0.6, vmax=1.0, alpha=0.8)
    ax.set_xlim(*CONUS_LON); ax.set_ylim(*CONUS_LAT)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"ANTHEIA-Scalar\n{label}", fontsize=9, fontweight="bold")

plt.colorbar(sc, ax=axes[-1], label="P(interaction)", shrink=0.8)
fig.suptitle(f"{PLANT} — Spatial Interaction Probability Maps\n"
             "(Note: logistic compression makes seasonal variation subtle — see PPE Δ maps below)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_predict_proba_maps.png", dpi=150, bbox_inches="tight", facecolor="white")
print("Saved fig_predict_proba_maps.png")

In [ ]:
# ── APPROACH 2 (PRIMARY): PPE Δ Hovmöller + combined figure ─────────────────
# Visualizes raw temporal overlap signal before classifier compression

CMAP_DELTA = "YlOrRd"
CMAP_DIFF  = "RdBu_r"

# Hovmöller: latitude band × week
delta_df["lat_band"] = np.floor(delta_df["lat"]).astype(int)
pred_df["lat_band"]  = np.floor(pred_df["lat"]).astype(int)
hovmoller_delta = delta_df.groupby(["lat_band", "week"])["delta"].mean().unstack(fill_value=np.nan)
hovmoller_a3    = pred_df.groupby(["lat_band",  "week"])["pred_a3"].mean().unstack(fill_value=np.nan)

# Latitude profile for Spatial Baseline
a2_profile = pred_df.groupby("lat")["pred_a2"].mean().sort_index()

# Mean A3 - A2 difference
diff_mean = pred_df.groupby(["lat", "lon"]).apply(
    lambda x: (x["pred_a3"] - x["pred_a2"]).mean()
).reset_index(name="diff_mean")

print("Building combined figure...")

fig = plt.figure(figsize=(26, 16), facecolor="white")
gs = gridspec.GridSpec(
    3, 7,
    height_ratios=[1.2, 1.2, 1.0],
    width_ratios=[1, 1, 1, 1, 1, 1, 0.05],
    hspace=0.45, wspace=0.15
)

month_ticks  = [0, 4, 8, 13, 17, 21, 26, 30, 34, 39, 43, 47, 51]
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec",""]
SPATIAL_WEEKS = {12: "Spring (Wk12)", 25: "Summer (Wk25)", 38: "Fall (Wk38)", 51: "Winter (Wk51)"}

# ── Row 0: Ground Truth | Spatial Baseline | PPE Δ × 4 seasons ───────────────

ax_gt = fig.add_subplot(gs[0, 0])
ax_gt.set_facecolor("white")
ax_gt.scatter(gt["decimalLongitude"], gt["decimalLatitude"],
              c="#2d6a4f", s=5, alpha=0.5)
ax_gt.set_xlim(*CONUS_LON); ax_gt.set_ylim(*CONUS_LAT)
ax_gt.set_aspect("equal"); ax_gt.set_xticks([]); ax_gt.set_yticks([])
ax_gt.set_title("Ground Truth\n(GloBI)", fontsize=10, fontweight="bold")

ax_a2 = fig.add_subplot(gs[0, 1])
ax_a2.set_facecolor("white")
a2_data = pred_df[pred_df["week"] == 0]
ax_a2.scatter(a2_data["lon"], a2_data["lat"],
              c=a2_data["pred_a2"], cmap=CMAP_PRED,
              s=5, vmin=0.6, vmax=1.0, alpha=0.8)
ax_a2.set_xlim(*CONUS_LON); ax_a2.set_ylim(*CONUS_LAT)
ax_a2.set_aspect("equal"); ax_a2.set_xticks([]); ax_a2.set_yticks([])
ax_a2.set_title("Spatial Baseline\n(Static)", fontsize=10, fontweight="bold")

delta_max = delta_df["delta"].max()
for col, (week, label) in enumerate(SPATIAL_WEEKS.items()):
    ax = fig.add_subplot(gs[0, col + 2])
    ax.set_facecolor("white")
    week_data = delta_df[delta_df["week"] == week]
    ax.scatter(week_data["lon"], week_data["lat"],
               c=week_data["delta"], cmap=CMAP_DELTA,
               s=5, vmin=0, vmax=delta_max, alpha=0.8)
    ax.set_xlim(*CONUS_LON); ax.set_ylim(*CONUS_LAT)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"PPE Δ\n{label}", fontsize=10, fontweight="bold")

# ── Row 1: Spatial Baseline latitude profile | PPE Δ Hovmöller ───────────────

ax_prof = fig.add_subplot(gs[1, :2])
ax_prof.set_facecolor("white")
ax_prof.barh(a2_profile.index, a2_profile.values,
             color="#2d6a4f", alpha=0.7, height=0.8)
ax_prof.axvline(a2_profile.mean(), color="red", linestyle="--", linewidth=1, alpha=0.7)
ax_prof.set_xlabel("Mean P(interaction)", fontsize=9)
ax_prof.set_ylabel("Latitude (°N)", fontsize=9)
ax_prof.set_title("Spatial Baseline — Latitude Profile\n(Static: no temporal variation)",
                  fontsize=10, fontweight="bold")
ax_prof.set_xlim(0.6, 1.0)
ax_prof.set_ylim(24, 50)

ax_hov = fig.add_subplot(gs[1, 2:6])
ax_hov.set_facecolor("white")
im_hov = ax_hov.imshow(
    hovmoller_delta.sort_index(ascending=False),
    aspect="auto", cmap=CMAP_DELTA,
    vmin=0, vmax=delta_max,
    extent=[0, 51, hovmoller_delta.index.min() - 0.5, hovmoller_delta.index.max() + 0.5]
)
ax_hov.set_xlabel("Month", fontsize=9)
ax_hov.set_ylabel("Latitude (°N)", fontsize=9)
ax_hov.set_title(
    "PPE Δ = min(f_curve, a_curve) — Latitude × Week (Hovmöller)\n"
    "Peak overlap propagates northward May–July, concentrated 35–45°N",
    fontsize=10, fontweight="bold"
)
ax_hov.set_xticks(month_ticks)
ax_hov.set_xticklabels(month_labels, fontsize=8)

cax_hov = fig.add_subplot(gs[1, 6])
plt.colorbar(
    plt.cm.ScalarMappable(cmap=CMAP_DELTA, norm=mcolors.Normalize(0, delta_max)),
    cax=cax_hov, label="PPE Δ"
)

# ── Row 2: Mean A3 − A2 difference map ───────────────────────────────────────

ax_diff = fig.add_subplot(gs[2, :6])
ax_diff.set_facecolor("white")
sc_diff = ax_diff.scatter(
    diff_mean["lon"], diff_mean["lat"],
    c=diff_mean["diff_mean"], cmap=CMAP_DIFF,
    s=6, vmin=-0.3, vmax=0.3, alpha=0.9
)
ax_diff.set_xlim(*CONUS_LON); ax_diff.set_ylim(*CONUS_LAT)
ax_diff.set_aspect("equal"); ax_diff.set_xticks([]); ax_diff.set_yticks([])
ax_diff.set_title(
    "Mean ANTHEIA-Scalar − Spatial Baseline Difference (Full Year)\n"
    "Negative (blue) = ANTHEIA lowers probability where phenology mismatches",
    fontsize=10, fontweight="bold"
)

cax_diff = fig.add_axes([0.92, 0.08, 0.015, 0.22])
plt.colorbar(
    plt.cm.ScalarMappable(cmap=CMAP_DIFF, norm=mcolors.Normalize(-0.3, 0.3)),
    cax=cax_diff, label="ΔP (ANTHEIA-Scalar − Spatial Baseline)"
)

fig.patch.set_facecolor("white")
fig.suptitle(
    f"{PLANT} — ANTHEIA Interaction Predictions",
    fontsize=15, fontweight="bold", y=1.01
)

plt.savefig(OUT_DIR / "fig_antheia_combined.png",
            dpi=150, bbox_inches="tight", facecolor="white")
print("Saved fig_antheia_combined.png")